# FinDisputeEval — NeMo Data Designer 客製 recipe v1（dispute-intake 多輪對話試產）

**位置**：計畫書 v10 §7.4 / Dataset Reference v2 行動項 No.59–61。
接在 `dataset/curated/seed_pools/cfpb_dispute/seed_v04`（1,838 正式 seed，經人工審查覆蓋）之後，是第一個真正的**合成生成** notebook——
取代煙霧測試用的官方 stock `multi_turn_chat.py`，改為 FinDispute 專用配方。

**設計（seed 合流 schema 的 v1 實作）**
- **內容錨**：CFPB seed（`relabel_required=False` 子集，標籤已淨的 1,599 筆）→ 逐列 seed dataset
- **結構桶**：ABCD EDA 分桶 → `structure_bucket` sampler（封閉詞彙表，不需逐列 seed）
- **情緒軌跡**：EmoWOZ EDA 分桶 → `emotion_trajectory` sampler
- **語言現象**：negation / hedging / escalation 以 Bernoulli sampler 釘在 **population proxy 實測率**
  （0.8664 / 0.1489 / 0.2888——直接從 `cfpb_reference_distributions.json` 讀，非手寫）
- **試產規模**：75 筆（母體比例分層配額），跑 §8.13 gate 通過後才放量

**輸入**（來自 `dataset/curated/seed_pools/cfpb_dispute/seed_v04/`，manifest 以 sha256 鏈回）
`cfpb_seed_pool.jsonl`、`cfpb_reference_distributions.json`、`seed_selection_manifest.json`

**輸出**（→ `outputs/generation/benchmark_v01/nemo_data_designer/seeded_dialogue/recipe_v01/<run_id>/`）
`pilot_seed_pool.parquet`、`dispute_dialogues_pilot_v1.jsonl`、`nemo_pilot_manifest.json`

**紅線**
1. 產物含 CFPB 原始敘事 → 只放 Drive／本地，**永不進 GitHub repo**（本地已有 `.gitignore` 防護）
2. seed 的 `claim_type` 只在 `relabel_required=False` 時視為可信 gold（v4 審查政策）
3. English-only；生成對話不得含真實 PII／真實公司名（seed 的 `company` 欄刻意不進 prompt）
4. fork 紀律：本 notebook 不覆寫任何舊版

**驗收清單**
- [ ] Step 0–2：輸入三檔 hash 記入 manifest；model alias 健康檢查通過
- [ ] Step 3–4：`relabel_required=False` = 1,599；pilot 配額表 ≈ 母體比例（fraud_scam 兩格為 0）
- [ ] Step 6：preview 2 筆人工目測（角色交替、佔位符已具體化、無 Reg E/Z 條文引用）
- [ ] Step 7–8：75 筆生成，`dialogue_valid` ≥ 95%、輪次合規 ≥ 95%、seed 覆蓋無缺漏
- [ ] Step 9：§8.13 gate——三現象率 + token 長度 JS 全 PASS（zelle 率僅回報）
- [ ] Step 10：manifest 含 `parent_manifest_sha256`；judge 分數分布回報後人工抽 10 筆對照

**成本概估**：75 ×（1 次 `nvidia-reasoning` 結構化生成 + 1 次 `nvidia-text` judge）+ preview 2 筆，
在 build.nvidia.com 免費額度內。

## Step 0. 環境、路徑、常數、manifest 起手

Colab：讀寫 Drive `FinDisputeEval/`；Data Designer 工作目錄放 `/content`（ephemeral）。
本地：以 cwd 為基準。`SEED=20260703` 與前序 notebook 一致。

In [ ]:
import sys, json, hashlib, random
from pathlib import Path
from datetime import datetime, timezone

NOTEBOOK_VERSION = 1
SEED = 20260703
PILOT_RECORDS = 75
DIALOGUE_MODEL_ALIAS = "nvidia-reasoning"   # nemotron-3-super-120b：多約束結構化生成
JUDGE_MODEL_ALIAS = "nvidia-text"           # nemotron-3-nano-30b：評分用，便宜
random.seed(SEED)

def progress(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

IN_COLAB = "google.colab" in sys.modules


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the FinDisputeEval project root.")


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/FinDisputeEval")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())

BASE_DIR = PROJECT_ROOT
V4_DIR = PROJECT_ROOT / "dataset" / "curated" / "seed_pools" / "cfpb_dispute" / "seed_v04"
RUN_ID = globals().get("RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
RUN_ROOT = PROJECT_ROOT / "outputs" / "generation" / "benchmark_v01" / "nemo_data_designer" / "seeded_dialogue" / "recipe_v01"
OUT_DIR = RUN_ROOT / f"run_{RUN_ID}_{'colab' if IN_COLAB else 'local'}"
ARTIFACT_DIR = Path("/content/dd_artifacts") if IN_COLAB else OUT_DIR / "artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SEED_POOL = V4_DIR / "cfpb_seed_pool.jsonl"
REF_DIST = V4_DIR / "cfpb_reference_distributions.json"
PARENT_MANIFEST = V4_DIR / "seed_selection_manifest.json"
for p in (SEED_POOL, REF_DIST, PARENT_MANIFEST):
    assert p.exists(), f"missing input: {p}"

manifest = {
    "notebook": "FinDisputeEval_NeMoDataDesigner_SeededDialogue_exp_001_colab",
    "notebook_version": NOTEBOOK_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "model_aliases": {"dialogue": DIALOGUE_MODEL_ALIAS, "judge": JUDGE_MODEL_ALIAS},
    "inputs": {p.name: {"sha256": sha256_file(p)} for p in (SEED_POOL, REF_DIST, PARENT_MANIFEST)},
    "parent_manifest_sha256": sha256_file(PARENT_MANIFEST),
    "stages": {}, "outputs": {},
}
progress(f"Step 0 complete: V4_DIR={V4_DIR} | OUT_DIR={OUT_DIR}")

## Step 1. 安裝依賴

只補缺的套件（不 `-U`、不動 requests——v2 修正 #7 的教訓）。spaCy/nltk 只有 Step 9 gate 用；
本地 Python ≥3.13 可能沒有 spaCy wheel，屆時 Step 9 請改在 Colab 跑。

In [ ]:
import importlib.util, subprocess

def _missing(mods):
    return [m for m in mods if importlib.util.find_spec(m) is None]

need = _missing(["data_designer", "pyarrow", "spacy", "nltk"])
pipmap = {"data_designer": "data-designer"}
if need:
    pkgs = [pipmap.get(m, m) for m in need]
    progress(f"installing: {pkgs}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

GATE_DEPS_OK = not _missing(["spacy", "nltk"])
if GATE_DEPS_OK:
    import spacy
    try:
        spacy.load("en_core_web_sm")
    except OSError:
        subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=True)
    import nltk
    try:
        nltk.data.find("sentiment/vader_lexicon.zip")
    except LookupError:
        nltk.download("vader_lexicon", quiet=True)
else:
    print("[warn] spaCy/nltk 不可用（本地新版 Python 常見）——Step 9 gate 請在 Colab 執行")
progress(f"Step 1 complete: gate_deps={GATE_DEPS_OK}")

## Step 2. NVIDIA_API_KEY 與 model alias 健康檢查

金鑰只存在本次 runtime。煙霧測試學到的坑：舊版 `~/.data-designer` 快取可能缺 alias →
偵測到缺就整個重生（`rm -rf` 等效）再讀一次。

In [ ]:
import os, shutil
from getpass import getpass

if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass("NVIDIA_API_KEY: ")

from data_designer.interface import DataDesigner
import data_designer.config as dd

def _engine_and_aliases():
    eng = DataDesigner(artifact_path=str(ARTIFACT_DIR))
    return eng, {c.alias for c in eng.get_default_model_configs()}

engine, aliases = _engine_and_aliases()
if not {DIALOGUE_MODEL_ALIAS, JUDGE_MODEL_ALIAS} <= aliases:
    shutil.rmtree(Path.home() / ".data-designer", ignore_errors=True)
    engine, aliases = _engine_and_aliases()
assert {DIALOGUE_MODEL_ALIAS, JUDGE_MODEL_ALIAS} <= aliases, f"aliases available: {aliases}"
progress(f"Step 2 complete: {DIALOGUE_MODEL_ALIAS} / {JUDGE_MODEL_ALIAS} ready")

## Step 3. 載入 v4 seed pool，取 `relabel_required=False` 子集

只信任已淨標籤（v4 審查政策）：fraud_scam 兩格整格排除，待剩餘 239 筆人工重標完成後解鎖。
展平 `linguistic` 巢狀欄；**刻意排除** `company`（真實公司名不進 prompt）與 `narrative_raw`
（`narrative_norm` 已是生成錨點）。預期 1,599 筆／14 格。

In [ ]:
import pandas as pd

progress("Step 3/9: loading v4 seed pool")
rows = []
with open(SEED_POOL, encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))
pool = pd.DataFrame(rows)
n_total = len(pool)

clean = pool[~pool["relabel_required"]].copy()
lin = pd.json_normalize(clean["linguistic"]).set_index(clean.index)
clean["seed_n_tokens"] = lin["n_tokens"]
clean["seed_neg_count"] = lin["neg_count"]
clean["seed_hedge_hits"] = lin["hedge_hits"]
clean["seed_esc_hits"] = lin["esc_hits"]
clean["seed_vader"] = lin["vader_compound"]
clean["amounts_str"] = clean["amounts"].map(lambda xs: ", ".join(f"${a}" for a in xs))

SEED_COLS = ["seed_id", "card_type", "claim_type", "route_hint", "zelle_mention",
             "anchor_confidence", "quality_score", "amounts_str", "narrative_norm",
             "seed_n_tokens", "seed_neg_count", "seed_hedge_hits", "seed_esc_hits", "seed_vader"]
seed_df = clean[SEED_COLS].reset_index(drop=True)

manifest["stages"]["seed_pool_rows"] = int(n_total)
manifest["stages"]["relabel_clean_rows"] = int(len(seed_df))
print(seed_df.groupby(["card_type", "claim_type"]).size().to_string())
progress(f"Step 3 complete: {len(seed_df):,}/{n_total:,} seeds（relabel_required=False）")

## Step 4. 母體比例分層抽 pilot（手寫治理邏輯，per library decision log）

配額 ∝ `population_proxy_inscope_2025plus` 的加權格子分布，限縮到可用 14 格後重新正規化：
- fraud_scam 兩格不在子集 → 自然為 0；`prepaid|transaction_banking_error` 無母體權重 → 0
- 最大餘數法；有母體權重的格子至少 1；格內以 `random_state=SEED` 抽樣（保留格內 zelle 混合比）

pilot parquet 就是 Data Designer 的 seed dataset（每列 1:1 生成一筆對話）。

In [ ]:
progress("Step 4/9: stratified pilot sampling")
ref = json.loads(REF_DIST.read_text(encoding="utf-8"))
POP = ref["population_proxy_inscope_2025plus"]
PHEN = POP["phenomenon_rates"]
pop_cells = POP["cell_distribution_weighted"]

avail = seed_df.groupby(["card_type", "claim_type"]).size()
cells = list(avail.index)
w = {c: float(pop_cells.get(f"{c[0]}|{c[1]}", 0.0)) for c in cells}
tot = sum(w.values())
share = {c: w[c] / tot for c in cells}

alloc = {c: min(int(share[c] * PILOT_RECORDS), int(avail[c])) for c in cells}
for c in cells:
    if share[c] > 0:
        alloc[c] = max(alloc[c], 1)
rem = {c: share[c] * PILOT_RECORDS - alloc[c] for c in cells}
while sum(alloc.values()) < PILOT_RECORDS:
    c = max((x for x in cells if alloc[x] < avail[x]), key=lambda x: rem[x])
    alloc[c] += 1; rem[c] -= 1
while sum(alloc.values()) > PILOT_RECORDS:
    c = max((x for x in cells if alloc[x] > (1 if share[x] > 0 else 0)), key=lambda x: -rem[x])
    alloc[c] -= 1; rem[c] += 1

parts = []
for c, k in alloc.items():
    if k == 0:
        continue
    grp = seed_df[(seed_df["card_type"] == c[0]) & (seed_df["claim_type"] == c[1])]
    parts.append(grp.sample(n=k, random_state=SEED))
pilot_df = pd.concat(parts).sort_values(["card_type", "claim_type", "seed_id"]).reset_index(drop=True)

PILOT_PARQUET = OUT_DIR / "pilot_seed_pool.parquet"
pilot_df.to_parquet(PILOT_PARQUET, index=False)
manifest["stages"]["pilot_rows"] = int(len(pilot_df))
manifest["pilot_cell_allocation"] = {f"{c[0]}|{c[1]}": int(k) for c, k in sorted(alloc.items())}
manifest["outputs"][PILOT_PARQUET.name] = {"sha256": sha256_file(PILOT_PARQUET), "rows": int(len(pilot_df))}
print(pd.Series(manifest["pilot_cell_allocation"]).to_string())
print(f"zelle seeds in pilot: {int(pilot_df['zelle_mention'].sum())}")
progress(f"Step 4 complete: pilot={len(pilot_df)} -> {PILOT_PARQUET.name}")

## Step 5a. 結構桶、情緒軌跡、對話 schema、prompt

**錨定與非錨定要分清楚（§8.13 誠實聲明 18）**：
- 三個語言現象的 Bernoulli `p` **直接取自 population proxy 實測率**——這部分是錨定的
- `structure_bucket` / `emotion_trajectory` 名稱對齊 ABCD / EmoWOZ EDA 分桶，但**權重是設計選擇**
  ——CFPB 單向敘事無法錨定輪次結構；EmoWOZ 的 abusive 桶刻意不生成
- 客戶端總字數目標近似 population proxy token 分位（118 / 197 / 314 / 578）

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

P_NEG = round(PHEN["negation_present"], 4)
P_HEDGE = round(PHEN["hedging_present"], 4)
P_ESC = round(PHEN["escalation_language_present"], 4)

STRUCTURE_BUCKETS = {                       # 權重＝設計選擇（非 CFPB 錨定）
    "straightforward_intake": 0.30, "slot_collection_heavy": 0.20, "policy_denial": 0.12,
    "escalation_to_supervisor": 0.13, "repair_misunderstanding": 0.15, "multi_issue": 0.10,
}
EMOTION_TRAJECTORIES = {                    # 權重＝設計選擇；不含 abusive
    "neutral_calm": 0.20, "frustrated_but_cooperative": 0.30, "dissatisfied_repair": 0.15,
    "anxious_fearful": 0.15, "angry_escalating": 0.20,
}
N_TURNS_VALUES, N_TURNS_WEIGHTS = [6, 8, 10, 12], [0.25, 0.35, 0.25, 0.15]
CUST_LEN_VALUES, CUST_LEN_WEIGHTS = [120, 200, 320, 560], [0.25, 0.40, 0.25, 0.10]

class Turn(BaseModel):
    role: Literal["customer", "agent"] = Field(description="Speaker of this turn.")
    content: str = Field(description="Utterance text, English only.")

class DisputeIntakeDialogue(BaseModel):
    turns: list[Turn] = Field(description="Alternating turns, customer speaks first.")

print(f"phenomenon anchors: neg={P_NEG} hedge={P_HEDGE} esc={P_ESC}")
progress("Step 5a complete: taxonomies + schema defined")

In [ ]:
SYSTEM_PROMPT_DIALOGUE = """You are an expert dialogue writer creating synthetic evaluation data for a bank's \
consumer-dispute intake assistant. You write realistic, natural multi-turn conversations between a bank customer \
and a dispute-intake agent. You never copy the source narrative verbatim; you translate it into a live conversation. \
All output must be in English. Never include real person names, account numbers, card numbers, SSNs, emails, or \
phone numbers; invent plausible generic details instead (e.g., "my Visa ending 4821" is fine)."""

PROMPT_DIALOGUE = """\
Write a multi-turn dispute-intake conversation between a bank CUSTOMER and an intake AGENT.

## Scenario facts (ground truth -- the conversation must stay consistent with these)
- Product bucket: {{ card_type }} (credit_reg_z = credit card; debit_reg_e = debit card / checking account; \
p2p_reg_e = P2P transfer service such as Zelle; prepaid_reg_e = prepaid card)
- Dispute category: {{ claim_type }}
- Dollar amounts involved: {{ amounts_str if amounts_str else "not specified -- invent one plausible amount" }}
- Source complaint narrative (the customer's real situation, written after the fact):
```
{{ narrative_norm }}
```
The narrative uses redaction placeholders: [DATE] = a date, [AMOUNT] = a dollar amount, [REDACTED] = a name or
identifier. Replace every placeholder concept with concrete plausible values (dates in 2025, amounts consistent
with the list above, generic merchant names). Call the bank "Meridian Bank".

## Conversation requirements
- Exactly {{ n_turns }} turns total, strictly alternating, starting with the customer.
- The customer's opening turn states the problem but NOT every detail; the agent elicits the rest
  (what happened, when, the amount, how the card/account was used, whether the customer authorized it).
- The customer's turns together should total roughly {{ customer_token_target }} words.
- Dialogue structure style: {{ structure_bucket }}
  (straightforward_intake = cooperative, linear slot filling;
   slot_collection_heavy = customer gives fragmented info, agent must ask many follow-up questions;
   policy_denial = the agent must explain that something the customer wants cannot be done at intake, customer pushes back;
   escalation_to_supervisor = customer demands a supervisor or threatens external escalation late in the dialogue;
   repair_misunderstanding = the agent misunderstands one detail mid-dialogue and the customer corrects it;
   multi_issue = customer raises one secondary concern besides the main dispute; the agent keeps intake on track)
- Customer emotional trajectory: {{ emotion_trajectory }}
  (neutral_calm = businesslike throughout;
   frustrated_but_cooperative = audibly frustrated yet answers everything;
   dissatisfied_repair = starts upset about prior bad service, gradually settles as the agent handles it well;
   anxious_fearful = worried about losing money, repeatedly seeks reassurance;
   angry_escalating = anger builds as the conversation progresses)
{% if include_negation %}- The customer naturally uses negation when denying or clarifying (e.g., "I never authorized this", "that wasn't me", "I did not receive a refund").
{% endif %}{% if include_hedging %}- The customer hedges about at least one detail (e.g., "I think it was around...", "I'm not sure exactly when...").
{% endif %}{% if include_escalation %}- At some point the customer mentions possible escalation (e.g., filing a CFPB complaint, contacting a lawyer, or asking for a supervisor).
{% endif %}{% if zelle_mention %}- The dispute involves Zelle and the customer refers to Zelle by name.
{% endif %}
## Agent behavior
- Professional, empathetic, concise. Confirms understanding, collects intake details, explains next steps and
  provisional timelines in plain language.
- The agent must NOT make a final liability decision, must NOT promise a refund outcome, and must NOT cite
  regulation numbers or statutes (no "Reg E" / "Reg Z" / "EFTA" citations) -- plain-language process descriptions only.
"""

PROMPT_JUDGE = """\
Evaluate this synthetic dispute-intake conversation.

## Scenario facts the conversation must respect
- Product bucket: {{ card_type }}; dispute category: {{ claim_type }}; amounts: {{ amounts_str }}
- Source narrative:
```
{{ narrative_norm }}
```

## Generation constraints it was asked to satisfy
- {{ n_turns }} alternating turns starting with the customer; customer turns total roughly {{ customer_token_target }} words
- structure = {{ structure_bucket }}; emotion trajectory = {{ emotion_trajectory }}
- required flags: negation={{ include_negation }}, hedging={{ include_hedging }}, \
escalation_mention={{ include_escalation }}, zelle={{ zelle_mention }}
- the agent must not decide liability, promise outcomes, or cite regulations

## Conversation to evaluate
{{ dispute_dialogue }}
"""
progress("Step 5a complete: prompts defined")

## Step 5b. 組 Data Designer config

seed dataset 用 `ORDERED` + `num_records = len(pilot)` → 每列 seed 恰生成一筆（可覆蓋驗證）。
sampler 全走 library（category with weights / Bernoulli）；LLM 欄兩支：結構化對話 + judge。

In [ ]:
progress("Step 5b/9: building Data Designer config")
cb = dd.DataDesignerConfigBuilder()
cb.with_seed_dataset(dd.LocalFileSeedSource(path=str(PILOT_PARQUET)),
                     sampling_strategy=dd.SamplingStrategy.ORDERED)

def cat_column(name, mapping):
    vals = list(mapping)
    return dd.SamplerColumnConfig(
        name=name, sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(values=vals, weights=[mapping[v] for v in vals]))

cb.add_column(cat_column("structure_bucket", STRUCTURE_BUCKETS))
cb.add_column(cat_column("emotion_trajectory", EMOTION_TRAJECTORIES))
cb.add_column(dd.SamplerColumnConfig(
    name="n_turns", sampler_type=dd.SamplerType.CATEGORY,
    params=dd.CategorySamplerParams(values=N_TURNS_VALUES, weights=N_TURNS_WEIGHTS)))
cb.add_column(dd.SamplerColumnConfig(
    name="customer_token_target", sampler_type=dd.SamplerType.CATEGORY,
    params=dd.CategorySamplerParams(values=CUST_LEN_VALUES, weights=CUST_LEN_WEIGHTS)))
for nm, p in (("include_negation", P_NEG), ("include_hedging", P_HEDGE), ("include_escalation", P_ESC)):
    cb.add_column(dd.SamplerColumnConfig(
        name=nm, sampler_type=dd.SamplerType.BERNOULLI, params=dd.BernoulliSamplerParams(p=p)))

cb.add_column(dd.LLMStructuredColumnConfig(
    name="dispute_dialogue", model_alias=DIALOGUE_MODEL_ALIAS,
    system_prompt=SYSTEM_PROMPT_DIALOGUE, prompt=PROMPT_DIALOGUE,
    output_format=DisputeIntakeDialogue))

cb.add_column(dd.LLMJudgeColumnConfig(
    name="generation_judge", model_alias=JUDGE_MODEL_ALIAS, prompt=PROMPT_JUDGE,
    scores=[
        dd.Score(name="seed_faithfulness",
                 description="Consistency with the scenario facts and source narrative.",
                 options={5: "fully consistent, no contradicted facts",
                          4: "one peripheral detail inconsistent",
                          3: "minor inconsistencies in peripheral details",
                          2: "one core fact (amount / product / what happened) contradicted",
                          1: "multiple core facts contradicted or scenario ignored"}),
        dd.Score(name="instruction_compliance",
                 description="Turn count, alternation, customer-first, requested phenomena flags, "
                             "structure and emotion styles all respected.",
                 options={5: "all constraints satisfied", 4: "one minor constraint missed",
                          3: "two constraints missed", 2: "several constraints missed",
                          1: "constraints largely ignored"}),
        dd.Score(name="naturalness",
                 description="Reads like a real customer-service conversation (colloquial customer, "
                             "professional agent), not like a written complaint being recited.",
                 options={5: "indistinguishable from a real transcript", 4: "mostly natural",
                          3: "somewhat stilted or templated", 2: "clearly synthetic",
                          1: "incoherent or narrative pasted into turns"}),
    ]))

cfg = cb.build()
manifest["stages"]["config_columns"] = [c.name for c in cb.get_column_configs()]
print("columns:", manifest["stages"]["config_columns"])
progress("Step 5b complete: config built")

## Step 6. Preview 2 筆（人工目測點）

目測重點：角色交替且客戶開場、佔位符已具體化（不得殘留 `[DATE]`）、agent 無條文引用、
現象旗標有兌現。不滿意就改 Step 5a 的 prompt 再 preview，**不要直接跑 Step 7**。

In [ ]:
RUN_PREVIEW = True
if RUN_PREVIEW:
    progress("Step 6/9: preview 2 records")
    pv = engine.preview(cb, num_records=2)
    try:
        pv.display_sample_record(0)
    except Exception as e:
        print(f"[warn] display_sample_record failed: {e}")
    if pv.dataset is not None and len(pv.dataset):
        rec = pv.dataset.iloc[0]
        d = rec["dispute_dialogue"]
        if isinstance(d, str):
            d = json.loads(d)
        print(f"--- seed={rec.get('seed_id')} cell={rec.get('card_type')}|{rec.get('claim_type')}")
        print(f"--- structure={rec.get('structure_bucket')} emotion={rec.get('emotion_trajectory')} "
              f"turns={rec.get('n_turns')} neg={rec.get('include_negation')}")
        for t in list(d.get("turns", []))[:6]:
            print(f"{t['role']:>9}: {t['content'][:150]}")
    progress("Step 6 complete: 人工目測後再進 Step 7")
else:
    progress("Step 6 skipped (RUN_PREVIEW=False)")

## Step 7. 試產 75 筆

冪等：輸出 JSONL 已存在就跳過（`FORCE_REGENERATE=True` 重跑）。
`num_records = len(pilot_df)` 配 `ORDERED` → seed 1:1。

In [ ]:
DIALOGUE_JSONL = OUT_DIR / "dispute_dialogues_pilot_v1.jsonl"
FORCE_REGENERATE = False

gen_df = None
if DIALOGUE_JSONL.exists() and not FORCE_REGENERATE:
    progress(f"Step 7 skipped: {DIALOGUE_JSONL.name} 已存在（cache hit）")
    manifest["stages"]["generation"] = "cache_hit"
else:
    progress(f"Step 7/9: generating {len(pilot_df)} dialogues（{DIALOGUE_MODEL_ALIAS}）…")
    results = engine.create(cb, num_records=len(pilot_df), dataset_name="dispute_pilot_v1")
    gen_df = results.load_dataset()
    manifest["stages"]["generation"] = f"created:{len(gen_df)}"
    progress(f"Step 7 complete: {len(gen_df)} records")

## Step 8. 解析、Pydantic 驗證、匯出 JSONL（→ Drive／本地，紅線 1）

每筆記錄 = seed 溯源欄 + sampler 值（＝gold 條件）+ 對話 + judge 分數。
`claim_type`／`card_type` 來自已淨 seed 標籤 → 可直接當下游 NLU 評測 gold；
對話輪次合規（交替、客戶開場、輪數）逐筆記錄，不硬砍。

In [ ]:
import numpy as np

def to_plain(v):
    if isinstance(v, np.ndarray):
        return [to_plain(x) for x in v.tolist()]
    if isinstance(v, np.generic):
        return v.item()
    if isinstance(v, dict):
        return {k: to_plain(x) for k, x in v.items()}
    if isinstance(v, list):
        return [to_plain(x) for x in v]
    return v

if gen_df is None:
    recs = [json.loads(l) for l in open(DIALOGUE_JSONL, encoding="utf-8")]
    progress(f"Step 8: reloaded {len(recs)} records from cache")
else:
    recs, n_bad = [], 0
    for _, row in gen_df.iterrows():
        r = {k: to_plain(v) for k, v in row.items()}
        d = r.get("dispute_dialogue")
        if isinstance(d, str):
            try:
                d = json.loads(d)
            except json.JSONDecodeError:
                d = None
        try:
            dlg = DisputeIntakeDialogue.model_validate(d)
            roles = [t.role for t in dlg.turns]
            r["dispute_dialogue"] = dlg.model_dump()
            r["dialogue_valid"] = True
            r["n_turns_actual"] = len(roles)
            r["starts_with_customer"] = bool(roles and roles[0] == "customer")
            r["alternates"] = all(a != b for a, b in zip(roles, roles[1:]))
        except Exception:
            n_bad += 1
            r["dialogue_valid"] = False
        recs.append(r)
    with open(DIALOGUE_JSONL, "w", encoding="utf-8") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")

    got = {r.get("seed_id") for r in recs}
    want = set(pilot_df["seed_id"])
    if got != want:
        print(f"[warn] seed coverage mismatch: missing={sorted(want - got)[:5]} extra={sorted(got - want)[:5]}")
    ok = [r for r in recs if r.get("dialogue_valid")]
    stats = {
        "records": len(recs), "dialogue_valid": len(ok), "invalid": n_bad,
        "starts_with_customer_rate": round(float(np.mean([r["starts_with_customer"] for r in ok])), 4) if ok else 0,
        "alternates_rate": round(float(np.mean([r["alternates"] for r in ok])), 4) if ok else 0,
        "turn_count_exact_rate": round(float(np.mean(
            [r["n_turns_actual"] == r.get("n_turns") for r in ok])), 4) if ok else 0,
        "seed_coverage_ok": got == want,
    }
    manifest["stages"]["dialogue_validation"] = stats
    manifest["outputs"][DIALOGUE_JSONL.name] = {"sha256": sha256_file(DIALOGUE_JSONL), "rows": len(recs)}
    print(json.dumps(stats, indent=2))
progress("Step 8 complete")

## Step 9. §8.13 試產 gate（對 `population_proxy_inscope_2025plus`）

**與 seed selection v3 完全同一套量測工具**（spaCy 佔位符合併、`dep_=="neg"`、同一份
escalation/hedging 詞典、同一 speech-act matcher、VADER）——量的是**合成對話的客戶端 utterance 層**。

門檻（試產啟發式，n=75 統計噪音大）：
- 三現象率：`|pilot − ref| ≤ max(0.10, 2.5·SE)` → PASS
- 客戶端 token 總長：以 ref 分位當 bin 的 JS divergence ≤ 0.1 → PASS
- zelle 率、VADER 分位、speech-act 分布：**僅回報**（zelle 由 seed 內容驅動，非生成自由度）

In [ ]:
progress("Step 9/9: §8.13 gate")
assert GATE_DEPS_OK, "spaCy/nltk 不可用——請在 Colab 跑 Step 9"
import math
import spacy
from spacy.language import Language
from spacy.matcher import Matcher, PhraseMatcher
from nltk.sentiment import SentimentIntensityAnalyzer

nlp = spacy.load("en_core_web_sm", disable=["ner"])
PLACEHOLDER_MATCHER = Matcher(nlp.vocab)
for _name in ("DATE", "AMOUNT", "REDACTED"):
    PLACEHOLDER_MATCHER.add(f"PH_{_name}", [[{"ORTH": "["}, {"ORTH": _name}, {"ORTH": "]"}]])

@Language.component("merge_placeholders")
def merge_placeholders(doc):
    spans = spacy.util.filter_spans([doc[s:e] for _, s, e in PLACEHOLDER_MATCHER(doc)])
    with doc.retokenize() as retok:
        for sp in spans:
            retok.merge(sp)
    return doc

if "merge_placeholders" not in nlp.pipe_names:
    nlp.add_pipe("merge_placeholders", first=True)

ESCALATION_TERMS = ["attorney", "lawyer", "sue", "lawsuit", "legal action", "small claims",
                    "cfpb", "ftc", "police report", "better business bureau", "attorney general",
                    "speak to a representative", "talk to a human", "supervisor", "regulator"]
HEDGING_TERMS = ["maybe", "might", "i think", "i believe", "not sure", "pretty sure",
                 "possibly", "perhaps", "i guess", "kind of", "sort of", "seems", "apparently"]
SPEECH_ACT_LEMMAS = ["dispute", "report", "demand", "request", "question", "complain", "refuse", "insist"]
pm_esc = PhraseMatcher(nlp.vocab, attr="LOWER")
pm_esc.add("ESC", [nlp.make_doc(t) for t in ESCALATION_TERMS])
pm_hedge = PhraseMatcher(nlp.vocab, attr="LOWER")
pm_hedge.add("HEDGE", [nlp.make_doc(t) for t in HEDGING_TERMS])
pm_zelle = PhraseMatcher(nlp.vocab, attr="LOWER")
pm_zelle.add("ZELLE", [nlp.make_doc("zelle")])
m_speech = Matcher(nlp.vocab)
m_speech.add("SPEECH_ACT", [[{"LEMMA": {"IN": SPEECH_ACT_LEMMAS}, "POS": "VERB"}]])
sia = SentimentIntensityAnalyzer()

valid = [r for r in recs if r.get("dialogue_valid")]
cust_texts = [" ".join(t["content"] for t in r["dispute_dialogue"]["turns"] if t["role"] == "customer")
              for r in valid]
grows = []
for doc, txt in zip(nlp.pipe(cust_texts, batch_size=32), cust_texts):
    grows.append({
        "n_tokens": len(doc),
        "neg": any(t.dep_ == "neg" for t in doc),
        "hedge": bool(pm_hedge(doc)), "esc": bool(pm_esc(doc)), "zelle": bool(pm_zelle(doc)),
        "vader": sia.polarity_scores(txt[:2000])["compound"],
        "speech_acts": sorted({doc[s:e].root.lemma_.lower() for _, s, e in m_speech(doc)}),
        "leftover_placeholder": ("[DATE]" in txt or "[AMOUNT]" in txt or "[REDACTED]" in txt),
    })
g = pd.DataFrame(grows)
n = len(g)

def gate_rate(name, got, ref_p):
    se = math.sqrt(max(ref_p * (1 - ref_p), 1e-9) / n)
    tol = max(0.10, 2.5 * se)
    return {"metric": name, "pilot": round(float(got), 4), "reference": round(float(ref_p), 4),
            "tolerance": round(tol, 4), "pass": bool(abs(got - ref_p) <= tol)}

checks = [
    gate_rate("negation_present", g["neg"].mean(), PHEN["negation_present"]),
    gate_rate("hedging_present", g["hedge"].mean(), PHEN["hedging_present"]),
    gate_rate("escalation_language_present", g["esc"].mean(), PHEN["escalation_language_present"]),
]

qs = POP["token_length_quantiles"]
bins = [0, qs["5"], qs["25"], qs["50"], qs["75"], qs["95"], float("inf")]
ref_hist = np.array([0.05, 0.20, 0.25, 0.25, 0.20, 0.05])
got_hist = np.histogram(g["n_tokens"], bins=bins)[0].astype(float)
got_hist = got_hist / max(1.0, got_hist.sum())

def js_divergence(p, q):
    p = np.asarray(p, dtype=float) + 1e-12
    q = np.asarray(q, dtype=float) + 1e-12
    p, q = p / p.sum(), q / q.sum()
    m = (p + q) / 2
    kl = lambda a, b: float((a * np.log2(a / b)).sum())
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

js_tok = js_divergence(got_hist, ref_hist)
checks.append({"metric": "customer_token_length_js", "pilot": round(js_tok, 4),
               "reference": 0.0, "tolerance": 0.1, "pass": bool(js_tok <= 0.1)})

gate = pd.DataFrame(checks)
print(gate.to_string(index=False))
print(f"[info] zelle_mention pilot={g['zelle'].mean():.4f} vs ref={PHEN['zelle_mention']:.4f}（seed 驅動，不設檻）")
print(f"[info] leftover placeholders: {int(g['leftover_placeholder'].sum())} 篇（應為 0）")
print(f"[info] customer token quantiles: {g['n_tokens'].quantile([.05, .25, .5, .75, .95]).round(0).to_dict()}")
print(f"[info] vader quantiles: {g['vader'].quantile([.05, .25, .5, .75, .95]).round(3).to_dict()}")
sa = pd.Series([v for lst in g["speech_acts"] for v in lst]).value_counts()
print(f"[info] speech acts: {sa.to_dict()}")

GATE_PASS = bool(gate["pass"].all())
manifest["gate"] = {"n_valid": int(n), "checks": checks,
                    "zelle_rate_info": round(float(g["zelle"].mean()), 4),
                    "leftover_placeholders": int(g["leftover_placeholder"].sum()),
                    "pass": GATE_PASS}
progress(f"Step 9 complete: GATE {'PASS' if GATE_PASS else 'FAIL'}")

## Step 10. Review judge-score distribution and persist the manifest

The LLM judge is a weak signal. An abnormal score distribution, such as many faithfulness scores at or below 3, means the generation prompt in Step 5a needs revision.

The final quality decision depends on a manual review of 10 generated dialogues. The run manifest links to `dataset/curated/seed_pools/cfpb_dispute/seed_v04/seed_selection_manifest.json` through `parent_manifest_sha256`.


In [ ]:
def judge_scores(r):
    j = r.get("generation_judge")
    if isinstance(j, str):
        try:
            j = json.loads(j)
        except json.JSONDecodeError:
            return {}
    if not isinstance(j, dict):
        return {}
    out = {}
    for key in ("seed_faithfulness", "instruction_compliance", "naturalness"):
        v = j.get(key)
        if isinstance(v, dict):
            v = v.get("score", v.get("value"))
        if v is not None:
            try:
                out[key] = int(v)
            except (TypeError, ValueError):
                pass
    return out

jrows = [judge_scores(r) for r in recs if r.get("dialogue_valid")]
jdf = pd.DataFrame([j for j in jrows if j])
if len(jdf):
    print(jdf.describe().round(2).to_string())
    dist = {c: jdf[c].value_counts().sort_index().to_dict() for c in jdf.columns}
    manifest["judge_score_distributions"] = {c: {int(k): int(v) for k, v in d.items()} for c, d in dist.items()}
else:
    print("[warn] no judge scores parsed — 檢查 generation_judge 欄位結構")

MANIFEST_PATH = OUT_DIR / "nemo_pilot_manifest.json"
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print(f"manifest -> {MANIFEST_PATH}")
print(json.dumps({k: manifest[k] for k in ("stages", "pilot_cell_allocation") if k in manifest},
                 indent=2, ensure_ascii=False, default=str)[:1500])
progress(f"Step 10 complete: GATE_PASS={manifest.get('gate', {}).get('pass')}")

## 誠實限制與放量條件

**這次試產「已錨定」的**：內容（CFPB 已淨 seed）、三現象率（population proxy 實測）、
客戶端長度目標（token 分位）、格子配比（母體比例）。

**仍是設計聲明、非錨定的**：
1. `structure_bucket` / `emotion_trajectory` 權重——CFPB 錨不了輪次結構（誠實聲明 18），
   ABCD/EmoWOZ 是跨 domain proxy，gap 須在報告明示
2. judge 分數 = LLM 自評，未經人工校準
3. n=75 → 現象率 95% CI 半寬約 ±0.08~0.11，gate 是方向性檢查非統計檢定
4. fraud_scam 兩格（母體佔比 ~14.8%）因標籤未淨而缺席——**放量前解鎖條件＝剩餘 239 筆重標完成**

**放量路徑**：gate PASS + 人工抽 10 筆對照 OK → 500 筆（同 recipe、每批重跑 gate）→ 2,000 筆；
屆時把 `PILOT_RECORDS` 改掉即可，配額邏輯與 manifest 鏈不變。